# Modul 1 — Baze de date globale deschise

**DELTA-Hub Workshop · 22 mai 2026**

În acest notebook rulăm exemple minime pe Copernicus CDS, Google Earth Engine și Microsoft Planetary Computer, focalizate pe Delta Dunării.

> 📌 **Important.** Salvează o copie în Drive (`File → Save a copy in Drive`) înainte să modifici, altfel pierzi munca.

## Setup (rulează prima dată, ~1 min)

Versiuni fixate ca să avem același mediu pentru toți participanții.

In [ ]:
!pip install -q \
    "cdsapi>=0.7" \
    "earthengine-api>=1.0" \
    "geemap" \
    "pystac-client>=0.8" \
    "planetary-computer>=1.0" \
    "xarray>=2024.10" \
    "rioxarray>=0.17" \
    "netCDF4" \
    "cartopy"

print("✓ Pachete instalate.")

## 1. Google Earth Engine — compozit Sentinel-2 peste Sf. Gheorghe

Prima dată când rulezi, GEE va deschide un tab pentru autentificare. Aprobă, copiază codul, lipește-l înapoi.

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='your-gee-project-id')  # înlocuiește cu project ID-ul tău GEE

In [ ]:
import geemap

sf_gheorghe = ee.Geometry.Point(29.60, 44.90)

s2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2024-07-01', '2024-09-01')
    .filterBounds(sf_gheorghe)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
)

print(f'Scene găsite: {s2.size().getInfo()}')

median = s2.median().select(['B4', 'B3', 'B2'])

Map = geemap.Map(center=[44.90, 29.60], zoom=11)
Map.addLayer(median, {'min': 0, 'max': 3000}, 'S2 mediana iulie-august 2024')
Map

## 2. Microsoft Planetary Computer — căutare STAC

Avantajul Planetary Computer: nu ai nevoie de cont pentru *căutare*, doar pentru descărcări intensive.

In [ ]:
import pystac_client
import planetary_computer

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)

search = catalog.search(
    collections=['sentinel-2-l2a'],
    bbox=[28.5, 44.5, 30.0, 45.5],  # Delta Dunării
    datetime='2024-07-01/2024-09-01',
    query={'eo:cloud_cover': {'lt': 20}},
)

items = list(search.items())
print(f'Găsite {len(items)} scene Sentinel-2 peste Delta.')
for item in items[:5]:
    print(f'  • {item.datetime.date()} — nori: {item.properties["eo:cloud_cover"]:.1f}%')

## 3. Copernicus CDS — ERA5 (opțional, necesită API key)

Înregistrare la <https://cds.climate.copernicus.eu>. Du-te la profilul tău și copiază API key-ul, apoi rulează celula de mai jos.

In [ ]:
import os

# ÎNLOCUIEȘTE cu propria cheie:
CDS_KEY = ''  # ex: 'aaaa-bbbb-cccc-dddd'

with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write(f'url: https://cds.climate.copernicus.eu/api\n')
    f.write(f'key: {CDS_KEY}\n')

print('✓ Credențiale CDS scrise în ~/.cdsapirc')

In [ ]:
import cdsapi

c = cdsapi.Client()
c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'variable': ['10m_u_component_of_wind', '10m_v_component_of_wind'],
        'year': '2024',
        'month': '07',
        'day': [f'{d:02d}' for d in range(1, 32)],
        'time': [f'{h:02d}:00' for h in range(24)],
        'area': [46, 28, 43, 32],  # N, W, S, E — Marea Neagră NV
        'format': 'netcdf',
    },
    'era5_winds_blacksea_jul2024.nc',
)

## Exercițiu

Alege una dintre cele trei platforme și extrage **temperatura medie a aerului peste Sulina** pentru luna iulie 2024. Adu rezultatul (un număr + linia de cod) pentru discuție.

*Hint:* Sulina ≈ 45.15° N, 29.65° E. Pe GEE, caută colecția `ECMWF/ERA5_LAND/HOURLY` și banda `temperature_2m`.